In [1]:
# @title 1. Dependencias, Autenticación y Funciones ETL (Soporta TXT, PDF, JPG, PNG)
!pip install -q google-genai pydantic gspread google-auth

import os
import shutil
from datetime import datetime
from google.colab import drive, auth, userdata
import gspread
from google.auth import default
from google import genai
from google.genai import types
from pydantic import BaseModel, Field

# 1. Autenticación y Montaje de Drive
drive.mount('/content/drive')
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

# 2. Inicializar cliente Gemini 3.6 Flash
api_key = userdata.get('GEMINI_API_KEY')
client = genai.Client(api_key=api_key)

# 3. Definición de Rutas en Drive e ID del Spreadsheet
BASE_PATH = "/content/drive/MyDrive/Logistics_Shipment_Tracker"
RAW_PATH = os.path.join(BASE_PATH, "01_raw_inputs")
ARCHIVE_PATH = os.path.join(BASE_PATH, "02_processed_archive")
SPREADSHEET_ID = "1FVs4iMi0O2UgRD_Z-ALbFnl6ZUtdaI_x--q1ztcsts0"

# 4. Esquema Estructurado de Extracción (INCLUYE CAMPO PARA WHATSAPP)
class ShippingData(BaseModel):
    tracking_number: str = Field(description="Número de guía o rastreo principal (Master Tracking ID o PRO Number)")
    carrier: str = Field(description="Empresa transportadora (ej. FedEx, UPS, USPS, TForce, SAIA, DHL)")
    shipper_name: str = Field(description="Nombre de la empresa o persona que envía (Remitente)", default="N/A")
    origin_city_state: str = Field(description="Ciudad y Estado de origen (ej. Costa Mesa, CA)", default="N/A")
    recipient_name: str = Field(description="Nombre del destinatario o consignatario", default="N/A")
    destination_city_state: str = Field(description="Ciudad y Estado de destino (ej. Pembroke Park, FL)", default="N/A")
    status: str = Field(description="Estado normalizado: LABEL_CREATED, IN_TRANSIT, OUT_FOR_DELIVERY, DELIVERED, EXCEPTION")
    estimated_delivery: str = Field(description="Fecha estimada de entrega en formato YYYY-MM-DD", default="N/A")
    weight_lbs: float = Field(description="Peso total del paquete/envío en libras (LB)", default=0.0)
    package_count: str = Field(description="Cantidad de bultos/piezas (ej. 1 of 2)", default="1 of 1")
    source_type: str = Field(description="Tipo de fuente: SCREENSHOT_WHATSAPP, PDF_LABEL, TEXT_EMAIL_OR_CHAT")
    client_whatsapp_message: str = Field(
        description="Redacta un mensaje profesional en inglés listo para enviar al cliente por WhatsApp. Incluye emojis, transportadora, número de tracking, estado actual, origen/destino y la URL directa de rastreo del carrier."
    )

# 5. Función de Procesamiento Multimodal + Texto (.txt)
def process_file_with_gemini(file_path: str) -> ShippingData:
    ext = os.path.splitext(file_path)[1].lower()

    # Manejo de Texto Plano (.txt)
    if ext == '.txt':
        source_type = 'TEXT_EMAIL_OR_CHAT'
        with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
            text_content = f.read()

        prompt = f"""
        Analiza detalladamente este texto de confirmación de envío/tracking o chat.
        Extrae la información según el esquema requerido.
        Tipo de fuente: {source_type}.
        Texto a analizar:
        \"\"\"
        {text_content}
        \"\"\"
        """

        response = client.models.generate_content(
            model="gemini-3.6-flash",
            contents=prompt,
            config=types.GenerateContentConfig(
                response_mime_type="application/json",
                response_schema=ShippingData,
                temperature=0.1
            )
        )
        return ShippingData.model_validate_json(response.text)

    # Manejo de Imágenes y PDFs
    elif ext in ['.jpg', '.jpeg', '.png']:
        mime_type = 'image/jpeg' if ext != '.png' else 'image/png'
        source_type = 'SCREENSHOT_WHATSAPP'
    elif ext == '.pdf':
        mime_type = 'application/pdf'
        source_type = 'PDF_LABEL'
    else:
        raise ValueError(f"Formato no soportado: {ext}")

    with open(file_path, "rb") as f:
        file_bytes = f.read()

    prompt = f"""
    Analiza detalladamente esta imagen o documento de tracking logístico.
    Extrae la información según el esquema requerido y genera el mensaje de WhatsApp para el cliente.
    El tipo de fuente original es: {source_type}.
    Si el documento contiene un Master Tracking Number o múltiples piezas, extrae el Master y el peso global.
    """

    response = client.models.generate_content(
        model="gemini-3.6-flash",
        contents=[
            types.Part.from_bytes(data=file_bytes, mime_type=mime_type),
            prompt
        ],
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=ShippingData,
            temperature=0.1
        )
    )

    return ShippingData.model_validate_json(response.text)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# @title 2. Ejecución del Pipeline ETL (Con reintentos automáticos para Gemini)
import requests
import time

def process_with_retry(temp_path, retries=3, delay=5):
    """Intenta procesar el archivo con Gemini; si recibe error 503, reintenta automáticamente."""
    for attempt in range(1, retries + 1):
        try:
            return process_file_with_gemini(temp_path)
        except Exception as e:
            if "503" in str(e) or "UNAVAILABLE" in str(e):
                print(f"⚠️ Servidor ocupado (503). Reintentando en {delay} segundos... (Intento {attempt}/{retries})")
                time.sleep(delay)
                delay *= 2  # Aumenta el tiempo de espera entre intentos
            else:
                raise e
    raise Exception("El servicio de Gemini no estuvo disponible tras varios reintentos.")

def run_pipeline_from_tally():
    try:
        sh = gc.open_by_key(SPREADSHEET_ID)
        ws_tally = sh.worksheet("Logistics Command Center - Ingesta")
        ws_analytics = sh.worksheet("tracking_analytics")
    except Exception as e:
        print(f"❌ Error al abrir las pestañas en la hoja de cálculo: {e}")
        return

    records = ws_tally.get_all_records()

    if not records:
        print("📁 No hay envíos registrados en Tally para procesar.")
        return

    print(f"🚀 Se encontraron {len(records)} registro(s) en la pestaña de Tally...\n")

    for idx, row in enumerate(records, start=2):
        file_url = row.get("Adjuntar PDF o Imagen (FedEx / UPS / Screenshot)", "")
        text_input = row.get("O Pegar Texto / Email del Proveedor", "")

        if not file_url and not text_input:
            continue

        print(f"⏳ Procesando envío de la fila {idx}...")

        try:
            # 1. Caso A: Si enviaron un archivo (Imagen / PDF)
            if file_url:
                response = requests.get(file_url)
                if response.status_code == 200:
                    temp_path = "/content/temp_tally_file"
                    temp_path += ".pdf" if ".pdf" in file_url.lower() else ".png"

                    with open(temp_path, "wb") as f:
                        f.write(response.content)

                    # Procesar con reintento automático
                    data = process_with_retry(temp_path)

                    if os.path.exists(temp_path):
                        os.remove(temp_path)
                else:
                    raise Exception(f"No se pudo descargar el archivo de Tally (HTTP {response.status_code})")

            # 2. Caso B: Si enviaron únicamente Texto Plano
            elif text_input:
                temp_path = "/content/temp_tally_text.txt"
                with open(temp_path, "w", encoding="utf-8") as f:
                    f.write(text_input)

                data = process_with_retry(temp_path)

                if os.path.exists(temp_path):
                    os.remove(temp_path)

            # 3. Guardar en 'tracking_analytics'
            processed_at = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            analytics_row = [
                data.tracking_number,
                data.carrier,
                data.shipper_name,
                data.origin_city_state,
                data.recipient_name,
                data.destination_city_state,
                data.status,
                data.estimated_delivery,
                data.weight_lbs,
                data.package_count,
                data.source_type,
                getattr(data, 'client_whatsapp_message', 'N/A'),
                processed_at
            ]

            ws_analytics.append_row(analytics_row)
            print(f"✅ Registrado exitosamente en 'tracking_analytics': {data.tracking_number} ({data.carrier})\n")

        except Exception as e:
            print(f"❌ Error procesando la fila {idx}: {str(e)}\n")

# Ejecutar el Pipeline
run_pipeline_from_tally()

🚀 Se encontraron 1 registro(s) en la pestaña de Tally...

⏳ Procesando envío de la fila 2...
✅ Registrado exitosamente en 'tracking_analytics': 874801202857 (FedEx)

